In [1]:
import requests
seg_uuid = "4895fc96-dca9-11ed-a241-00155d0d192b"
segmentation = requests.get(f"http://127.0.0.1:8008/api/segmentations/{seg_uuid}/")
segmentation = segmentation.json()
utterances = segmentation["utterance_set"]
text = " ".join([u["text"] for u in utterances])

In [3]:
import spacy
from fastcoref import spacy_component
import requests

PORT = 8008
SERVER = "http://127.0.0.1"

podcasts = requests.get(f"{SERVER}:{PORT}/api/podcasts/")
podcasts = podcasts.json()
for i, p in enumerate(podcasts):
    print(f"{i}: {p['title']}")

0: Verdict with Ted Cruz
1: Ta Kommandoen med Geir Aker
2: Misjonen med Antonsen og Golden
3: Leger om livet
4: Norsken, svensken og dansken
5: Burde vært pensum
6: Huberman Lab
7: Stuff You Should Know
8: The Ben Shapiro Show
9: The Megyn Kelly Show
10: Pivot
11: The Daily
12: The Ezra Klein Show
13: Lex Fridman Podcast
14: Checks and Balance from The Economist
15: Money Talks from The Economist
16: The Economist Asks
17: The Ramsey Show
18: Dateline NBC
19: Pod Save America
20: Freakonomics Radio


In [5]:
import spacy
from fastcoref import spacy_component
import requests

PORT = 8008
SERVER = "http://127.0.0.1"

podcasts = requests.get(f"{SERVER}:{PORT}/api/podcasts/")
podcasts = podcasts.json()

for podcast in podcasts[10:]:
    if not podcast["language"].startswith("en"):
        print(f"skipping {podcast['title']}, language is {podcast['language']}")
        continue
    for audioitem in podcast['audioitem_set']:
        #if audioitem['title'] != "Ep. 1707 - World Famous YouTuber MrBeast Hit With Trans Controversy":
        #    continue
        print("starting episode: ", audioitem['title'], podcast['title'])
        for transcription in audioitem['transcription_set']:
            for segmentation in transcription['segmentation_set']:
                if segmentation['name'] == "spaCy":
                    nlp = spacy.load("en_core_web_lg")
                    nlp.add_pipe(
                        "fastcoref", 
                        config={'model_architecture': 'LingMessCoref', 'model_path': 'biu-nlp/lingmess-coref', 'device': 'cuda:0'}
                    )
                    seg_uuid = segmentation['uuid']
                    segmentation = requests.get(f"{SERVER}:{PORT}/api/segmentations/{seg_uuid}/")
                    segmentation = segmentation.json()
                    utterances = segmentation["utterance_set"]
                    # add a context field to each utterance with the 20 previous utterances
                    for i, utt in enumerate(utterances):
                        utt["context"] = " ".join([u["text"] for u in utterances[max(0, i-100):i+1]])

                    docs = nlp.pipe(
                    [utterance["context"] for utterance in utterances],
                    component_cfg={"fastcoref": {'resolve_text': True}}
                    )
                    try:
                        docs = list(docs)
                    except:
                        print("error with ", audioitem['title'], podcast['title'])
                        for doc in docs:
                            print(doc)
                        continue

                    assert len(docs) == len(utterances)

                    for i, doc in enumerate(docs):
                        resolved_text = doc._.resolved_text

                        # Create a new Doc object without running the entire pipeline
                        sentences_doc = nlp.make_doc(resolved_text)

                        # Apply the "senter" component to the sentences_doc
                        nlp.get_pipe("senter")(sentences_doc)

                        first = coref = next(sentences_doc.sents)
                        for coref in sentences_doc.sents: pass

                        first = original = next(doc.sents)
                        for original in doc.sents: pass
                        if coref.text.strip() != original.text.strip():
                            # write to API
                            utt_uuid = utterances[i]["uuid"]
                            utterances[i]["text_coref"] = coref.text.strip()
                            # no changes to these child record, so remove them
                            utterances[i].pop("classification_set")
                            utterances[i].pop("query_set")
                            res = requests.post(f"{SERVER}:{PORT}/api/utterances/{utt_uuid}/", json=utterances[i])
                            print(res.status_code)

                    

starting episode:  Blue Checks Live On (For Now), Adam Neumann of Arabia, and Guest Liz Hoffman Pivot


04/21/2023 02:05:46 - INFO - 	 missing_keys: []
04/21/2023 02:05:46 - INFO - 	 unexpected_keys: []
04/21/2023 02:05:46 - INFO - 	 mismatched_keys: []
04/21/2023 02:05:46 - INFO - 	 error_msgs: []
04/21/2023 02:05:46 - INFO - 	 Model Parameters: 590.0M, Transformer: 434.6M, Coref head: 155.4M
04/21/2023 02:06:01 - INFO - 	 Tokenize 256 inputs...
04/21/2023 02:06:07 - INFO - 	 ***** Running Inference on 256 texts *****
Inference: 100%|██████████| 256/256 [00:33<00:00,  7.60it/s]
04/21/2023 02:07:00 - INFO - 	 Tokenize 256 inputs...
04/21/2023 02:07:08 - INFO - 	 ***** Running Inference on 256 texts *****
Inference: 100%|██████████| 256/256 [00:47<00:00,  5.42it/s]
04/21/2023 02:08:15 - INFO - 	 Tokenize 256 inputs...
04/21/2023 02:08:23 - INFO - 	 ***** Running Inference on 256 texts *****
Inference: 100%|██████████| 256/256 [00:45<00:00,  5.57it/s]
04/21/2023 02:09:34 - INFO - 	 Tokenize 256 inputs...
04/21/2023 02:09:44 - INFO - 	 ***** Running Inference on 256 texts *****
Inference: 1

200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200


04/21/2023 02:11:50 - INFO - 	 missing_keys: []
04/21/2023 02:11:50 - INFO - 	 unexpected_keys: []
04/21/2023 02:11:50 - INFO - 	 mismatched_keys: []
04/21/2023 02:11:50 - INFO - 	 error_msgs: []
04/21/2023 02:11:50 - INFO - 	 Model Parameters: 590.0M, Transformer: 434.6M, Coref head: 155.4M
04/21/2023 02:12:04 - INFO - 	 Tokenize 256 inputs...
04/21/2023 02:12:10 - INFO - 	 ***** Running Inference on 256 texts *****
Inference: 100%|██████████| 256/256 [00:32<00:00,  7.83it/s]
04/21/2023 02:12:56 - INFO - 	 Tokenize 256 inputs...
04/21/2023 02:13:02 - INFO - 	 ***** Running Inference on 256 texts *****
Inference: 100%|██████████| 256/256 [00:29<00:00,  8.63it/s]
04/21/2023 02:13:53 - INFO - 	 Tokenize 256 inputs...
04/21/2023 02:14:02 - INFO - 	 ***** Running Inference on 256 texts *****
Inference: 100%|██████████| 256/256 [00:54<00:00,  4.71it/s]
04/21/2023 02:15:16 - INFO - 	 Tokenize 256 inputs...
04/21/2023 02:15:25 - INFO - 	 ***** Running Inference on 256 texts *****
Inference: 1

error with  Trump Indicted, Alibaba Splits, School Shooting Misinformation Pivot
starting episode:  Twitter Blues, Israel Protests, and TikTok espionage with Emily Baker-White Pivot


...

200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200


04/21/2023 09:22:30 - INFO - 	 missing_keys: []
04/21/2023 09:22:30 - INFO - 	 unexpected_keys: []
04/21/2023 09:22:30 - INFO - 	 mismatched_keys: []
04/21/2023 09:22:30 - INFO - 	 error_msgs: []
04/21/2023 09:22:30 - INFO - 	 Model Parameters: 590.0M, Transformer: 434.6M, Coref head: 155.4M
04/21/2023 09:22:49 - INFO - 	 Tokenize 256 inputs...
04/21/2023 09:22:58 - INFO - 	 ***** Running Inference on 256 texts *****
Inference: 100%|██████████| 256/256 [00:52<00:00,  4.84it/s]
04/21/2023 09:24:15 - INFO - 	 Tokenize 256 inputs...
04/21/2023 09:24:25 - INFO - 	 ***** Running Inference on 256 texts *****
Inference: 100%|██████████| 256/256 [01:07<00:00,  3.79it/s]
04/21/2023 09:25:37 - INFO - 	 Tokenize 37 inputs...
04/21/2023 09:25:39 - INFO - 	 ***** Running Inference on 37 texts *****
Inference: 100%|██████████| 37/37 [00:11<00:00,  3.19it/s]


200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200


04/21/2023 09:26:01 - INFO - 	 missing_keys: []
04/21/2023 09:26:01 - INFO - 	 unexpected_keys: []
04/21/2023 09:26:01 - INFO - 	 mismatched_keys: []
04/21/2023 09:26:01 - INFO - 	 error_msgs: []
04/21/2023 09:26:01 - INFO - 	 Model Parameters: 590.0M, Transformer: 434.6M, Coref head: 155.4M
04/21/2023 09:26:20 - INFO - 	 Tokenize 256 inputs...
04/21/2023 09:26:27 - INFO - 	 ***** Running Inference on 256 texts *****
Inference: 100%|██████████| 256/256 [00:50<00:00,  5.05it/s]


error with  539. Why Does One Tiny State Set the Rules for Everyone? Freakonomics Radio


In [5]:
if coref.text.strip() != original.text.strip():
    # write to API
    utt_uuid = utterances[i]["uuid"]
    utterances[i]["text_coref"] = coref.text.strip()
    utterances[i].pop("classification_set")
    utterances[i].pop("query_set")
    res = requests.post(f"{SERVER}:{PORT}/api/utterances/{utt_uuid}/", json=utterances[i])
    print(res.status_code)



200


In [4]:
utterances[i]

{'hidden': False,
 'start': '1427.02',
 'end': '1434.26',
 'speaker': 'SPEAKER_03',
 'text': 'This is not a mob boss who is threatening retaliation against a prosecutor.',
 'text_coref': 'Donald Trump is not a mob boss who is threatening retaliation against a prosecutor.',
 'microfacts': None,
 'claimspan': None,
 'uuid': '79637fbe-dad4-11ed-ba56-00155d8020a1',
 'classification_set': [{'utterance': 427670,
   'qualifier': 'Checkworthiness',
   'label': '0.5511140742607193',
   'category': 'Checkworthy',
   'agent': 'ClaimBuster-BBA'},
  {'utterance': 427670,
   'qualifier': 'Checkworthiness',
   'label': '0',
   'category': 'Checkworthy',
   'agent': 'Factiverse'}],
 'query_set': [],
 'context': "877, the number 4 gold IRA. Or online at AugustaPreciousMetals.com. That's AugustaPreciousMetals.com. Use the promo code Ben and you will get 10 years of fees covered, up to 10 years, which is pretty awesome. AugustaPreciousMetals.com. Senator, I want to play for you this NBC News reporter bec

In [3]:
# add a context field to each utterance with the 20 previous utterances
for i, utt in enumerate(utterances):
    utt["context"] = " ".join([u["text"] for u in utterances[max(0, i-100):i+1]])

In [7]:
docs = nlp.pipe(
   [utterance["context"] for utterance in utterances],
   component_cfg={"fastcoref": {'resolve_text': True}}
)
docs = list(docs)

04/20/2023 17:15:23 - INFO - 	 Tokenize 256 inputs...
04/20/2023 17:15:33 - INFO - 	 ***** Running Inference on 256 texts *****
Inference: 100%|██████████| 256/256 [01:12<00:00,  3.52it/s]
04/20/2023 17:17:10 - INFO - 	 Tokenize 240 inputs...
04/20/2023 17:17:19 - INFO - 	 ***** Running Inference on 240 texts *****
Inference: 100%|██████████| 240/240 [01:00<00:00,  3.97it/s]


In [8]:
assert len(docs) == len(utterances)

In [11]:
for i, doc in enumerate(docs):
   resolved_text = doc._.resolved_text

   # Create a new Doc object without running the entire pipeline
   sentences_doc = nlp.make_doc(resolved_text)

   # Apply the "senter" component to the sentences_doc
   nlp.get_pipe("senter")(sentences_doc)

   first = coref = next(sentences_doc.sents)
   for coref in sentences_doc.sents: pass

   first = original = next(doc.sents)
   for original in doc.sents: pass
   if coref.text.strip() != original.text.strip():
       print(utterances[i]["uuid"])
       print(coref.text.strip())
       print(original.text.strip())
       print("\n")
   

4899d410-dca9-11ed-a241-00155d0d192b
Senator Ted Cruz, nice to chat with Senator Ted Cruz today.
Senator, nice to chat with you today.


489a7d7a-dca9-11ed-a241-00155d0d192b
And the big headline that so many people, especially in the Catholic faith, are talking about is the fact that it looks like the FBI was trying to infiltrate Catholic groups and trying to use people in the church to basically become, you know, the FBI informants.
And that is the fact that it looks like the FBI was trying to infiltrate Catholic groups and trying to use people in the church to basically become, you know, FBI informants.


489ac672-dca9-11ed-a241-00155d0d192b
There's also a lot of people that are asking the question now, did the FBI lie to Congress about this saying, I don't know, the FBI didn't do this?
There's also a lot of people that are asking the question now, did the FBI lie to Congress about this saying, I don't know, we didn't do this?


489b6fb4-dca9-11ed-a241-00155d0d192b
Senator Ted Cruz's

In [66]:
for sent in doc.sents:
    print(sent)

Have you got a small company or a small business?
FIKEN makes it easy to send invoices or offers.
A couple of clicks and it's sent and calculated.
You can collect invoices yourself or let FIKEN do it automatically for you.
You can also sell the invoices and get paid with the same amount or press a button to transfer to a cash register.
Do like over 70,000 others.
Take the calculation yourself with FIKEN.
Try for free on FIKEN.no.
Welcome.
It is Verdict with Senator Ted Cruz.
Ben Ferguson with you.
Senator, nice to chat with you today.
And let's start with the big headline that so many people, especially in the Catholic faith, are talking about.
And that is the fact that it looks like the FBI was trying to infiltrate Catholic groups and trying to use people in the church to basically become, you know, FBI informants.
There's also a lot of people that are asking the question now, did the FBI lie to Congress about this saying, I don't know, we didn't do this?
And now it's coming to light 

In [44]:
doc_list = list(docs)

04/20/2023 14:44:00 - INFO - 	 Tokenize 256 inputs...
04/20/2023 14:44:11 - INFO - 	 ***** Running Inference on 256 texts *****
Inference: 100%|██████████| 256/256 [01:11<00:00,  3.57it/s]
04/20/2023 14:45:45 - INFO - 	 Tokenize 240 inputs...
04/20/2023 14:45:54 - INFO - 	 ***** Running Inference on 240 texts *****
Inference: 100%|██████████| 240/240 [00:59<00:00,  4.03it/s]


In [47]:
doc_list[20]._.resolved_text

''

In [67]:
for sent in doc.sents:
    print(sent)
    print(sent._.doc_extensions["coref_clusters"])

Have you got a small company or a small business?
(None, None, None, None)
FIKEN makes it easy to send invoices or offers.
(None, None, None, None)
A couple of clicks and it's sent and calculated.
(None, None, None, None)
You can collect invoices yourself or let FIKEN do it automatically for you.
(None, None, None, None)
You can also sell the invoices and get paid with the same amount or press a button to transfer to a cash register.
(None, None, None, None)
Do like over 70,000 others.
(None, None, None, None)
Take the calculation yourself with FIKEN.
(None, None, None, None)
Try for free on FIKEN.no.
(None, None, None, None)
Welcome.
(None, None, None, None)
It is Verdict with Senator Ted Cruz.
(None, None, None, None)
Ben Ferguson with you.
(None, None, None, None)
Senator, nice to chat with you today.
(None, None, None, None)
And let's start with the big headline that so many people, especially in the Catholic faith, are talking about.
(None, None, None, None)
And that is the fact t